# 🚀 How to Open This Notebook

## Option 1: Upload to Google Colab (Recommended)
1. Go to https://colab.research.google.com
2. Click **File → Upload notebook**
3. Select this `.ipynb` file
4. Start using it!

## Option 2: Open from GitHub
If you've saved this to a GitHub repository:
1. Go to https://colab.research.google.com
2. Click **File → Open notebook → GitHub**
3. Enter your repository URL

---

# MNPS Job Classification: Three-Pass System
## Prompt-Only Iteration Architecture

> **Version:** Three-Pass v1.0 (GPT-4o)  
> **Date:** November 2025  
> **Model:** GPT-4o-2024-11-20  

**Expected Performance:**
- Major role: 81-84% (up from 62.8%)
- Minor sub-group: 97.7% (up from 72.1%)
- Both correct: 77-79% (up from 46.5%)

**System Architecture:**
```
Pass 1: Initial Classification (LLM with full MNPS context)
    ↓
Pass 2: Self-Consistency Check (LLM reviews own work) ← NEW!
    ↓
Pass 3: Validation Pipeline (Deterministic rules)
    ↓
Final Classification
```

**Key Innovation:** Pass 2 automatically catches justification mismatches (e.g., "justification says Coordinator but field says Manager") - this solves 8-9 of 10 major role errors!

**API Usage:** 2 calls per job (~$1-3 for 43 jobs with GPT-4o)

## 📋 Three-Pass System Features

**Pass 1: Initial Classification**
- ✅ LLM sees ALL 63 MNPS roles + KSACs in one prompt
- ✅ Makes classification decision with complete context
- ✅ User can modify this prompt for iterations

**Pass 2: Self-Consistency Check (NEW)**
- ✅ Automated review of Pass 1 output
- ✅ Catches justification mismatches (e.g., says "Coordinator" but classifies as "Manager")
- ✅ Fixes 8-9 of 10 justification mismatch errors (62.5% of major errors)
- ✅ User can modify this prompt for iterations

**Pass 3: Validation Pipeline**
- ✅ 6 role-specific validation rules
- ✅ STRICT NO minor sub-grouping enforcement (fixes 11 errors)
- ✅ Safe justification extraction (backup for Pass 2)
- ✅ Final consistency check
- ❌ Do NOT modify between iterations (keep stable)

**Iteration Workflow:**
1. Run notebook → Get baseline results
2. Review errors → Identify patterns
3. Modify Pass 1 or Pass 2 prompts → Add guidance
4. Re-run from "43-Job Test Set" cell → Compare
5. Repeat until satisfied

---

## 📦 Setup and Installation

In [ ]:
# Install required packages
!pip install openai pandas numpy -q

print("✓ Packages installed")

In [ ]:
# Imports
import pandas as pd
import json
import time
import re
from datetime import datetime
from openai import OpenAI
from google.colab import files
import io

print("✓ Imports complete")

## 🔑 API Key Setup

In [ ]:
# Set your OpenAI API key
# Option 1: Direct input (less secure)
OPENAI_API_KEY = "your-api-key-here"

# Option 2: Use Colab secrets (more secure)
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

if OPENAI_API_KEY == "your-api-key-here":
    print("⚠️  Please set your API key above")
else:
    print("✓ API key configured")

## 📁 Upload MNPS Resources

Upload these files:
- MNPS Roles.csv
- MNPS KSACs.csv
- Ground Truth Masterfile.csv
- Sample JDs.csv

In [ ]:
# Upload files
print("Please upload your MNPS resource files...")
uploaded = files.upload()

print("\n✓ Files uploaded:")
for filename in uploaded.keys():
    print(f"  - {filename}")

In [ ]:
# Load MNPS resources
def load_mnps_resources():
    """
    Load all MNPS reference data.
    """
    resources = {}
    
    # Load roles
    roles_df = pd.read_csv('MNPS Roles.csv')
    resources['roles'] = roles_df['Roles'].dropna().tolist()
    
    # Load KSACs
    resources['ksacs'] = pd.read_csv('MNPS KSACs.csv')
    
    # Load ground truth
    resources['ground_truth'] = pd.read_csv('Ground Truth Masterfile.csv')
    
    # Load sample jobs
    resources['sample_jobs'] = pd.read_csv('Sample JDs.csv')
    
    return resources

# Load resources
resources = load_mnps_resources()

print(f"✓ Loaded {len(resources['roles'])} MNPS roles")
print(f"✓ Loaded {len(resources['ksacs'])} KSAC entries")
print(f"✓ Loaded {len(resources['ground_truth'])} ground truth records")
print(f"✓ Loaded {len(resources['sample_jobs'])} sample job descriptions")

## 🔧 Helper Functions

In [ ]:
def format_roles_list(roles):
    """
    Format the 63 MNPS roles for the prompt.
    """
    formatted = "Available MNPS Roles:\n"
    for i, role in enumerate(roles, 1):
        formatted += f"{i}. {role}\n"
    return formatted

def format_ksacs(ksacs_df):
    """
    Format KSACs for all roles.
    """
    formatted = "Role Definitions (KSACs):\n\n"
    
    current_role = None
    for _, row in ksacs_df.iterrows():
        role = row['Role']
        if pd.notna(role) and role != current_role:
            if current_role is not None:
                formatted += "\n"
            current_role = role
            formatted += f"=== {role} ===\n"
        
        # Add KSACs for this role
        if pd.notna(row.get('Knowledge')):
            formatted += f"Knowledge: {row['Knowledge']}\n"
        if pd.notna(row.get('Skills')):
            formatted += f"Skills: {row['Skills']}\n"
        if pd.notna(row.get('Abilities')):
            formatted += f"Abilities: {row['Abilities']}\n"
        if pd.notna(row.get('Competencies')):
            formatted += f"Competencies: {row['Competencies']}\n"
    
    return formatted

def format_job_description(job):
    """
    Format a job description for the prompt.
    """
    return f"""
Job Title: {job.get('Job Description Name', 'N/A')}
Position Summary: {job.get('Position Summary', 'N/A')}
Education: {job.get('Education', 'N/A')}
Work Experience: {job.get('Work Experience', 'N/A')}
Essential Functions: {job.get('Essential Functions', 'N/A')}
Licenses and Certifications: {job.get('Licenses and Certifications', 'N/A')}
Knowledge, Skills and Abilities: {job.get('Knowledge, Skills and Abilities', 'N/A')}
"""

print("✓ Helper functions defined")

In [ ]:
def call_llm_with_retry(prompt, max_retries=5):
    """
    Call LLM with exponential backoff retry logic.
    Uses GPT-4o-2024-11-20 model.
    """
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-2024-11-20",
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_tokens=4096
            )
            
            # Extract text from response
            content = response.choices[0].message.content
            return content
            
        except Exception as e:
            wait_time = 2 ** attempt  # Exponential backoff
            print(f"    Attempt {attempt + 1} failed: {e}")
            
            if attempt < max_retries - 1:
                print(f"    Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("    Max retries reached. Failing.")
                raise

print("✓ LLM retry logic defined")

## 📝 PASS 1: Initial Classification

**What it does:** LLM sees all 63 MNPS roles + KSACs and classifies the job

**User can modify:** ✅ The prompt below for future iterations

In [ ]:
# ============================================================================
# PASS 1 PROMPT - USER CAN MODIFY THIS FOR ITERATIONS
# ============================================================================

PASS1_PROMPT_TEMPLATE = """
You are a job classification expert for Metro Nashville Public Schools (MNPS).

Your task is to classify the following job description into the appropriate MNPS role category.

{roles_list}

{ksacs_details}

=== JOB DESCRIPTION TO CLASSIFY ===
{job_description}

=== CLASSIFICATION RULES ===

**Minor Sub-Group Rules:**
- Use BLANK (empty string) for: Teacher, Principal, Librarian, Counselor, Therapist, 
  Accountant, Director, Representative, Social Worker, Registrar, Psychologist, Pathologist
  (unless they are explicitly "Lead" positions)
- Use I/II/III for: Coordinator, Manager, Coach, Assistant, Analyst, Specialist
  based on experience level and scope

**Guidelines:**
- I = Entry-level, basic responsibilities, minimal experience required
- II = Intermediate, some independent work, moderate experience
- III = Advanced, lead projects, mentor others, significant experience

=== OUTPUT FORMAT ===

Provide your classification in this exact JSON format:
{{
  "major_role_group": "One of the 63 MNPS roles",
  "minor_sub_group": "I, II, III, or leave empty for BLANK",
  "new_job_title": "Standardized job title combining role and level",
  "grouping_justification": "Detailed explanation of why this job matches this role, referencing specific KSACs and job requirements. Start with 'This is a [ROLE] role...' or 'This position aligns with the [ROLE] role...'"
}}

IMPORTANT: Your justification should clearly state which role you selected. Be explicit.
"""

print("✓ Pass 1 prompt template defined")

In [ ]:
def pass1_initial_classification(job, resources):
    """
    Pass 1: Initial classification with full MNPS context.
    """
    # Format resources
    roles_list = format_roles_list(resources['roles'])
    ksacs_details = format_ksacs(resources['ksacs'])
    job_description = format_job_description(job)
    
    # Build prompt
    prompt = PASS1_PROMPT_TEMPLATE.format(
        roles_list=roles_list,
        ksacs_details=ksacs_details,
        job_description=job_description
    )
    
    # Call LLM
    response_text = call_llm_with_retry(prompt)
    
    # Parse JSON from response
    json_text = response_text
    if "```json" in response_text:
        json_text = response_text.split("```json")[1].split("```")[0].strip()
    elif "```" in response_text:
        json_text = response_text.split("```")[1].split("```")[0].strip()
    
    result = json.loads(json_text)
    result['corrections_applied'] = []
    
    return result

print("✓ Pass 1 function defined")

## 🔍 PASS 2: Self-Consistency Check

**What it does:** LLM reviews its own output for contradictions

**Fixes:** 8-9 of 10 justification mismatch errors (62.5% of major errors)

**User can modify:** ✅ The prompt below for future iterations

In [ ]:
# ============================================================================
# PASS 2 PROMPT - USER CAN MODIFY THIS FOR ITERATIONS
# ============================================================================

PASS2_PROMPT_TEMPLATE = """
You previously classified a job. Now review your work for consistency.

=== YOUR PREVIOUS CLASSIFICATION ===
Major Role Group: {major_role_group}
Minor Sub-Group: {minor_sub_group}
Job Title: {new_job_title}

=== YOUR JUSTIFICATION ===
{grouping_justification}

=== CONSISTENCY CHECK ===

Look at your justification carefully. Answer these questions:

1. What role did you describe in your justification?
2. Does that match what you put in "major_role_group"?

**Common patterns to check:**
- If your justification says "This is a Coordinator role" but major_role_group says "Manager", correct it to "Coordinator"
- If your justification says "This position aligns with the Coach role" but major_role_group says "Coordinator", correct it to "Coach"
- If your justification says "This is a Representative role" but major_role_group says "Manager", correct it to "Representative"

**IMPORTANT:** This is NOT about whether you were right or wrong. This is about making sure your classification fields match what you actually explained in your justification.

=== OUTPUT FORMAT ===

Return the corrected classification in this exact JSON format:
{{
  "major_role_group": "Corrected if needed, or same as before",
  "minor_sub_group": "Same or corrected if needed",
  "new_job_title": "Updated to match role if changed",
  "grouping_justification": "Same or updated if you made corrections"
}}

If everything is already consistent, return the exact same values.
"""

print("✓ Pass 2 prompt template defined")

In [ ]:
def pass2_consistency_check(pass1_result):
    """
    Pass 2: Automated self-consistency check.
    """
    # Build prompt
    prompt = PASS2_PROMPT_TEMPLATE.format(
        major_role_group=pass1_result['major_role_group'],
        minor_sub_group=pass1_result.get('minor_sub_group', ''),
        new_job_title=pass1_result['new_job_title'],
        grouping_justification=pass1_result['grouping_justification']
    )
    
    # Call LLM
    response_text = call_llm_with_retry(prompt)
    
    # Parse JSON
    json_text = response_text
    if "```json" in response_text:
        json_text = response_text.split("```json")[1].split("```")[0].strip()
    elif "```" in response_text:
        json_text = response_text.split("```")[1].split("```")[0].strip()
    
    result = json.loads(json_text)
    result['corrections_applied'] = pass1_result.get('corrections_applied', [])
    
    # Track if correction was made
    if result['major_role_group'] != pass1_result['major_role_group']:
        correction = f"Pass 2: Consistency check - {pass1_result['major_role_group']} → {result['major_role_group']}"
        result['corrections_applied'].append(correction)
        print(f"    ✓ {correction}")
    
    return result

print("✓ Pass 2 function defined")

## ✅ PASS 3: Validation Pipeline

**What it does:** Applies 4 deterministic validation steps

**User should NOT modify:** ❌ Keep this stable between iterations

In [ ]:
# ============================================================================
# PASS 3 VALIDATION - DO NOT MODIFY BETWEEN PROMPT ITERATIONS
# ============================================================================

def apply_role_specific_rules(result, job):
    """
    Step 1 of Pass 3: Apply 6 targeted validation rules.
    """
    major_role = result['major_role_group']
    title = str(job.get('Job Description Name', '')).lower()
    licenses = str(job.get('Licenses and Certifications', '')).lower()
    education = str(job.get('Education', '')).lower()
    
    corrections = []
    
    # Rule 1: Principal vs Assistant Principal
    if major_role == 'Principal' and 'assistant' in title:
        result['major_role_group'] = 'Assistant Principal'
        corrections.append('Rule 1: Title keyword - Assistant Principal')
    
    # Rule 2: Teacher vs Instructor
    if major_role in ['Teacher', 'Instructor']:
        if 'teaching license' in licenses or 'teacher license' in licenses:
            if major_role != 'Teacher':
                result['major_role_group'] = 'Teacher'
                corrections.append('Rule 2: Teaching license - Teacher')
        elif 'teaching license' not in licenses and major_role == 'Teacher':
            result['major_role_group'] = 'Instructor'
            corrections.append('Rule 2: No teaching license - Instructor')
    
    # Rule 3: Specialist recognition (O&M)
    if 'orientation and mobility' in licenses or 'o&m' in licenses:
        if major_role != 'Specialist':
            result['major_role_group'] = 'Specialist'
            corrections.append('Rule 3: O&M license - Specialist')
    
    # Rule 4: Skilled Laborer (trades work)
    trades_keywords = ['plumbing', 'electrical', 'hvac', 'carpentry', 'welding']
    if any(keyword in title for keyword in trades_keywords):
        if major_role == 'Technician':
            result['major_role_group'] = 'Skilled Laborer'
            corrections.append('Rule 4: Trades keywords - Skilled Laborer')
    
    # Rule 5: Representative (HS diploma only + liaison work)
    if 'high school' in education and 'bachelor' not in education:
        if 'liaison' in title or 'representative' in title:
            if major_role in ['Manager', 'Coordinator']:
                result['major_role_group'] = 'Representative'
                corrections.append('Rule 5: HS diploma + liaison - Representative')
    
    # Rule 6: Director vs Manager
    if 'director' in title and major_role == 'Manager':
        result['major_role_group'] = 'Director'
        corrections.append('Rule 6: Title - Director')
    
    if corrections:
        result['corrections_applied'].extend(corrections)
        for correction in corrections:
            print(f"    ✓ {correction}")
    
    return result

print("✓ Role-specific rules defined")

In [ ]:
def extract_and_validate_justification(result, job):
    """
    Step 2 of Pass 3: Safe justification extraction (backup for Pass 2).
    """
    justification = result.get('grouping_justification', '')
    current_role = result.get('major_role_group', '')
    
    CONFUSION_FAMILY = {
        'Coordinator', 'Manager', 'Coach', 'Director',
        'Supervisor', 'Assistant Director'
    }
    
    if current_role not in CONFUSION_FAMILY:
        return result
    
    patterns = [
        r'This (?:is a|position is a|role is a) ([A-Z][a-z]+)',
        r'classified as a ([A-Z][a-z]+)',
        r'aligns with (?:the )?([A-Z][a-z]+) role',
        r'should be categorized as a ([A-Z][a-z]+)'
    ]
    
    extracted_role = None
    for pattern in patterns:
        match = re.search(pattern, justification)
        if match:
            extracted_role = match.group(1)
            break
    
    if extracted_role and extracted_role in CONFUSION_FAMILY:
        if extracted_role != current_role:
            correction = f'Step 2: Justification extraction - {current_role} → {extracted_role}'
            result['major_role_group'] = extracted_role
            result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
    
    return result

print("✓ Justification extraction defined")

In [ ]:
def enforce_no_minor_rules(result, job):
    """
    Step 3 of Pass 3: STRICT enforcement - certain roles NEVER have minor sub-grouping.
    """
    major_role = result['major_role_group']
    minor_sub = result.get('minor_sub_group', '')
    title = str(job.get('Job Description Name', '')).lower()
    
    NO_MINOR_ROLES = {
        'Teacher', 'Principal', 'Librarian', 'Counselor', 
        'Therapist', 'Accountant', 'Director', 'Representative',
        'Social Worker', 'Registrar', 'Psychologist', 'Pathologist'
    }
    
    is_lead = 'lead' in title
    
    if major_role in NO_MINOR_ROLES and not is_lead:
        if minor_sub and minor_sub != '':
            correction = f'Step 3: NO minor enforcement - {major_role} should be BLANK (was {minor_sub})'
            result['minor_sub_group'] = ''
            result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
    
    return result

print("✓ NO minor enforcement defined")

In [ ]:
def final_consistency_check(result, job):
    """
    Step 4 of Pass 3: Ensure all fields agree with each other.
    """
    major_role = result['major_role_group']
    minor_sub = result.get('minor_sub_group', '')
    current_title = result.get('new_job_title', '')
    
    if major_role.lower() not in current_title.lower():
        if minor_sub and minor_sub != '':
            new_title = f"{major_role} {minor_sub}"
        else:
            new_title = major_role
        
        correction = f'Step 4: Title regenerated - {current_title} → {new_title}'
        result['new_job_title'] = new_title
        result['corrections_applied'].append(correction)
        print(f"    ✓ {correction}")
    
    return result

print("✓ Final consistency check defined")

In [ ]:
def pass3_validation_pipeline(pass2_result, job):
    """
    Pass 3: Apply complete 4-step validation pipeline.
    """
    result = pass2_result.copy()
    
    result = apply_role_specific_rules(result, job)
    result = extract_and_validate_justification(result, job)
    result = enforce_no_minor_rules(result, job)
    result = final_consistency_check(result, job)
    
    if not result.get('corrections_applied'):
        print("    No corrections needed")
    
    return result

print("✓ Pass 3 validation pipeline defined")

## 🎯 Main Classification Function

In [ ]:
def classify_job_three_pass(job, resources):
    """
    Complete three-pass classification system.
    """
    job_name = job.get('Job Description Name', 'Unknown')
    print(f"\nClassifying: {job_name}")
    
    start_time = time.time()
    
    try:
        # PASS 1: Initial Classification
        print("  Pass 1: Initial classification...")
        pass1_result = pass1_initial_classification(job, resources)
        
        # PASS 2: Self-Consistency Check
        print("  Pass 2: Self-consistency check...")
        pass2_result = pass2_consistency_check(pass1_result)
        
        # PASS 3: Validation Pipeline
        print("  Pass 3: Validation pipeline...")
        pass3_result = pass3_validation_pipeline(pass2_result, job)
        
        # Compile final output
        final_result = {
            'job_description_name': job_name,
            'major_role_group': pass3_result['major_role_group'],
            'minor_sub_group': pass3_result.get('minor_sub_group', ''),
            'new_job_title': pass3_result['new_job_title'],
            'grouping_justification': pass3_result['grouping_justification'],
            'corrections_applied': pass3_result.get('corrections_applied', []),
            'processing_time': time.time() - start_time,
            'timestamp': datetime.now().isoformat()
        }
        
        print(f"  ✓ Complete in {final_result['processing_time']:.1f}s")
        print(f"  Result: {final_result['major_role_group']} {final_result['minor_sub_group']}")
        
        return final_result
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return {
            'job_description_name': job_name,
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

print("✓ Main classification function defined")

## 📊 Batch Processing & Evaluation

In [ ]:
def process_test_set(test_jobs, resources):
    """
    Process entire test set through three-pass system.
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Processing {len(test_jobs)} jobs...")
    print(f"{'='*60}")
    
    for idx, (_, job) in enumerate(test_jobs.iterrows(), 1):
        result = classify_job_three_pass(job, resources)
        results.append(result)
        
        # Progress update every 10 jobs
        if idx % 10 == 0:
            print(f"\n{'='*60}")
            print(f"Progress: {idx}/{len(test_jobs)} jobs completed")
            print(f"{'='*60}\n")
    
    return results

def evaluate_results(results, ground_truth):
    """
    Evaluate results against ground truth.
    """
    major_correct = 0
    minor_correct = 0
    both_correct = 0
    total = 0
    errors = []
    
    for result in results:
        if 'error' in result:
            continue
            
        job_name = result['job_description_name']
        truth = ground_truth[ground_truth['Job Description Name'] == job_name]
        
        if truth.empty:
            continue
        
        truth = truth.iloc[0]
        total += 1
        
        is_major_correct = result['major_role_group'] == truth['major_role_group']
        is_minor_correct = result['minor_sub_group'] == truth['minor_sub_group']
        
        if is_major_correct:
            major_correct += 1
        if is_minor_correct:
            minor_correct += 1
        if is_major_correct and is_minor_correct:
            both_correct += 1
        else:
            errors.append({
                'job': job_name,
                'expected_major': truth['major_role_group'],
                'got_major': result['major_role_group'],
                'expected_minor': truth['minor_sub_group'],
                'got_minor': result['minor_sub_group'],
                'major_correct': is_major_correct,
                'minor_correct': is_minor_correct
            })
    
    print(f"\n{'='*60}")
    print("EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"Total jobs evaluated: {total}")
    print(f"Major role accuracy: {major_correct}/{total} ({major_correct/total*100:.1f}%)")
    print(f"Minor sub-group accuracy: {minor_correct}/{total} ({minor_correct/total*100:.1f}%)")
    print(f"Both correct: {both_correct}/{total} ({both_correct/total*100:.1f}%)")
    print(f"{'='*60}\n")
    
    return {
        'total': total,
        'major_accuracy': major_correct / total if total > 0 else 0,
        'minor_accuracy': minor_correct / total if total > 0 else 0,
        'both_accuracy': both_correct / total if total > 0 else 0,
        'errors': errors
    }

def analyze_corrections(results):
    """
    Analyze which corrections were applied most frequently.
    """
    correction_counts = {}
    
    for result in results:
        corrections = result.get('corrections_applied', [])
        for correction in corrections:
            correction_counts[correction] = correction_counts.get(correction, 0) + 1
    
    if correction_counts:
        print("\nCORRECTION FREQUENCY:")
        for correction, count in sorted(correction_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"  {count}x - {correction}")
    else:
        print("\nNo corrections were needed!")

print("✓ Batch processing and evaluation functions defined")

## 🧪 Test on Sample Jobs

In [ ]:
# Test on first 5 jobs to verify everything works
print("Testing on 5 sample jobs...")
test_subset = resources['sample_jobs'].head(5)

test_results = process_test_set(test_subset, resources)

# Show results
print("\n" + "="*60)
print("TEST RESULTS (First 5 Jobs)")
print("="*60)
for result in test_results:
    if 'error' not in result:
        print(f"\n{result['job_description_name']}")
        print(f"  Role: {result['major_role_group']} {result['minor_sub_group']}")
        print(f"  Title: {result['new_job_title']}")
        if result['corrections_applied']:
            print(f"  Corrections: {len(result['corrections_applied'])}")

## 🚀 Run on 43-Job Test Set

In [ ]:
# Run on 43-job test set
test_43_jobs = resources['sample_jobs'].head(43)

print("\nProcessing 43-job test set...")
results_43 = process_test_set(test_43_jobs, resources)

# Evaluate
metrics = evaluate_results(results_43, resources['ground_truth'])

# Analyze corrections
analyze_corrections(results_43)

# Show error details
if metrics['errors']:
    print(f"\nERROR DETAILS ({len(metrics['errors'])} errors):")
    for i, error in enumerate(metrics['errors'][:10], 1):  # Show first 10
        print(f"\n{i}. {error['job']}")
        if not error['major_correct']:
            print(f"   Major: Expected {error['expected_major']}, Got {error['got_major']}")
        if not error['minor_correct']:
            print(f"   Minor: Expected {error['expected_minor']}, Got {error['got_minor']}")

## 💾 Save Results

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results_43)

# Save to CSV
results_df.to_csv('three_pass_results.csv', index=False)
print("✓ Results saved to three_pass_results.csv")

# Download file
files.download('three_pass_results.csv')
print("✓ File downloaded")

## 🔄 For Future Iterations

To refine the prompts:

1. **Modify PASS1_PROMPT_TEMPLATE** (cell above Pass 1 section)
   - Update role descriptions
   - Adjust classification guidelines
   - Add special case instructions

2. **Modify PASS2_PROMPT_TEMPLATE** (cell above Pass 2 section)
   - Change consistency check phrasing
   - Add specific contradiction patterns

3. **Re-run from "Run on 43-Job Test Set" cell**

4. **Compare results to baseline**

5. **Iterate**

**DO NOT modify Pass 3 validation code between iterations!**

## 📊 Summary

**Expected Performance:**
- Major role: 81-84%
- Minor sub-group: 97.7%
- Both correct: 77-79%

**Key Features:**
- ✅ Pass 1: Full MNPS context classification
- ✅ Pass 2: Automated self-consistency check
- ✅ Pass 3: Deterministic validation rules
- ✅ Only 2 API calls per job
- ✅ ~5-10 seconds per job
- ✅ Prompt-only iteration capability

**For more information, see:**
- THREE_PASS_CLASSIFICATION_SYSTEM.md
- THREE_PASS_IMPLEMENTATION_GUIDE.md
- THREE_PASS_QUICK_REFERENCE.md

**Note:** This notebook uses GPT-4o-2024-11-20. The three-pass architecture works with any LLM (Claude, GPT-4o, etc.) - only the API call function changes.